# Satisfactory Production Optimization — AWESOME Sink Points

**Goal:** Given fixed raw resource extraction rates, find the recipe throughputs
that maximize AWESOME Sink points per minute.

**Method:** Linear Programming. The math is unchanged from the project-parts version;
only the *objective coefficients* change — every manufacturable item now has a value
(its sink points), so the solver considers the entire production graph at once.


## 1. The Math

### Variables
Let $x_r \geq 0$ be the throughput of recipe $r$ in *buildings-at-100%-clock*.
A value of 2.5 means 2.5 machines running full-speed (or equivalently, 3 machines at 83%).

### Flow balance (constraint for every non-resource item $i$)

$$
\underbrace{\sum_{r:\,i\in\text{products}(r)} x_r \cdot \dot{p}_{r,i}}_{\text{production rate (items/min)}}
\;\geq\;
\underbrace{\sum_{r:\,i\in\text{inputs}(r)} x_r \cdot \dot{q}_{r,i}}_{\text{consumption rate (items/min)}}
$$

The **net output** of item $i$ is the difference — the rate at which $i$ accumulates
and can be fed into the AWESOME Sink.

### Resource constraints

$$
\sum_{r:\,s\in\text{inputs}(r)} x_r \cdot \dot{q}_{r,s} \leq S_s \quad \forall s \in \text{resources}
$$

### Objective — maximize sink points per minute

$$
\text{maximize} \sum_{i \notin \text{resources}} v_i \cdot \underbrace{\left(
  \sum_{r \to i} x_r \cdot \dot{p}_{r,i} - \sum_{r \leftarrow i} x_r \cdot \dot{q}_{r,i}
\right)}_{\text{net output rate of } i}
$$

where $v_i$ is the AWESOME Sink point value of item $i$ (0 for unsinkable items).

**Key insight:** Unlike the project-parts version where only ~12 items appeared in the
objective, here every item in the game has a coefficient — most are zero (unsinkable),
but those with positive values are all candidates. The solver weighs the resource cost
of making something expensive against the point reward.


In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import pulp
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display, HTML

from satisfactory import ITEMS, RECIPES_FOR, RESOURCES


## 2. Resource Supply

In [2]:
RESOURCE_SUPPLY: dict[str, float] = {
    'iron-ore':     5 * 480,   # 2,400/min
    'copper-ore':   2 * 480,   #   960/min
    'coal':         3 * 480,   # 1,440/min
    'limestone':    2 * 480,   #   960/min
    'crude-oil':    2 * 300,   #   600/min
    'water':        99_999,    # effectively unlimited
    'caterium-ore': 2 * 480,   #   960/min
}

df_supply = pd.DataFrame([
    {'Resource': k, 'Supply (items/min)': v if v < 90_000 else '∞'}
    for k, v in RESOURCE_SUPPLY.items()
])
display(df_supply.style.hide(axis='index'))


Resource,Supply (items/min)
iron-ore,2400
copper-ore,960
coal,1440
limestone,960
crude-oil,600
water,∞
caterium-ore,960


In [3]:
# Preview: top sinkable items by point value
sinkable = {k: v for k, v in ITEMS.items() if v.sink_points > 0 and not v.is_resource}
top = sorted(sinkable.values(), key=lambda i: i.sink_points, reverse=True)[:15]
df_top = pd.DataFrame([{'Item': i.name, 'Sink points': i.sink_points} for i in top])
print("Top sinkable items by point value:")
display(df_top.style.hide(axis='index'))


Top sinkable items by point value:


Item,Sink points
Ballistic Warp Drive,2895334
Thermal Propulsion Rocket,728508
AI Expansion Server,597652
Nuclear Pasta,538976
Assembly Director System,500176
Biochemical Sculptor,301778
Pressure Conversion Cube,255088
Neural-Quantum Processor,248034
Turbo Motor,240496
Plutonium Fuel Rod,153184


## 4. Building the LP

Same structure as before; only the objective changes.


## 5. Solve

In [4]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('.').resolve()))

from solve import solve

result     = solve(RESOURCE_SUPPLY)
status     = result["status"]
objective  = result["objective"]
active     = result["active"]
net_output = result["net"]
used       = result["used"]
shadow     = result["shadow"]

# Reconstruct all_recipes, all_items, non_resource_items
# so downstream cells (assertions, results tables, Sankey) continue unchanged
from satisfactory import RECIPES_FOR, RESOURCES
all_recipes = []
seen = set()
for recipes in RECIPES_FOR.values():
    for r in recipes:
        if not r.is_alt and r.key not in seen:
            all_recipes.append(r)
            seen.add(r.key)
all_items = set()
for r in all_recipes:
    all_items.update(r.ingredients)
    all_items.update(r.products)
non_resource_items = all_items - RESOURCES

# Compat dict for downstream cells that reference x[r.key]
_active_rates = {r.key: rate for r, rate in active}

print(f"Status:          {status}")
print(f"Sink points/min: {objective:,.0f}")


Status:          Optimal
Sink points/min: 835,077


## 6. Correctness Assertions

LP solvers can return infeasible or suboptimal results if the problem is mis-formulated.
These assertions verify the solution satisfies every property we care about.
Running this cell after every change to resources or the model gives instant feedback.

> **Satisfactory idiom:** In Satisfactory, no factory can consume more resources than it extracts,
> and no processing step can output more material than it receives. These assertions encode
> exactly those conservation laws.


In [5]:
# ── Assertion 1: solver found an optimal solution ─────────────────────────────
assert status == 'Optimal', (
    f"Solver did not find optimum: status = {status}\n"
    f"Check resource supplies and flow balance constraints."
)

# ── Assertion 2: no resource is over-extracted ────────────────────────────────
resource_errors = []
for resource in RESOURCES:
    supply = RESOURCE_SUPPLY.get(resource, 0)
    actual = used.get(resource, 0)
    if actual > supply + 1e-4:
        resource_errors.append(f"{resource}: used {actual:.2f} > supply {supply}")
assert not resource_errors, "Resource constraint(s) violated:\n" + "\n".join(resource_errors)

# ── Assertion 3: no item has negative net flow ────────────────────────────────
flow_errors = []
for item in non_resource_items:
    net = net_output(item)
    if net < -1e-4:
        flow_errors.append(f"{item}: net = {net:.4f}")
assert not flow_errors, "Flow balance violated:\n" + "\n".join(flow_errors)

# ── Assertion 4: objective matches manual recomputation ───────────────────────
manual_obj = sum(
    ITEMS[item].sink_points * net_output(item)
    for item in non_resource_items
    if item in ITEMS and ITEMS[item].sink_points > 0
)
lp_obj = objective
assert abs(manual_obj - lp_obj) < 1.0, (
    f"Objective mismatch: LP reported {lp_obj:,.2f}, manual = {manual_obj:,.2f}"
)

# ── Assertion 5 (Satisfactory-specific): iron ore must be used ───────────────
iron_consumed = used.get('iron-ore', 0)
if RESOURCE_SUPPLY.get('iron-ore', 0) > 0:
    assert iron_consumed > 0, "No iron ore consumed — LP ignoring a free resource."

print(f"All assertions passed.")
print(f"  Resources: all within supply limits")
print(f"  Flow balance: all items non-negative")
print(f"  Objective: LP {lp_obj:,.0f} == manual {manual_obj:,.0f}")
print(f"  Iron ore consumed: {iron_consumed:.1f}/min of {RESOURCE_SUPPLY.get('iron-ore',0)}/min available")


All assertions passed.
  Resources: all within supply limits
  Flow balance: all items non-negative
  Objective: LP 835,077 == manual 835,077
  Iron ore consumed: 2400.0/min of 2400/min available


## 7. Results

### What's being sunk and at what rate?

In [6]:
rows = []
for item in sorted(non_resource_items):
    if item not in ITEMS:
        continue
    pts = ITEMS[item].sink_points
    if pts == 0:
        continue
    net = net_output(item)
    sunk = max(0, net)
    if sunk < 0.01:
        continue
    rows.append({
        'Item': ITEMS[item].name,
        'Sunk (items/min)': round(sunk, 3),
        'Points each': pts,
        'Points/min': round(sunk * pts, 0),
    })

df_sink = (
    pd.DataFrame(rows)
    .sort_values('Points/min', ascending=False)
    .reset_index(drop=True)
)
total_pts = df_sink['Points/min'].sum()
display(df_sink.style.hide(axis='index').format({'Sunk (items/min)': '{:.3f}', 'Points/min': '{:,.0f}'}))
print(f"\nTotal: {total_pts:,.0f} sink points/min")


Item,Sunk (items/min),Points each,Points/min
Assembly Director System,1.190,500176,"595,448"
Magnetic Field Generator,16.098,11000,"177,074"
Stun Rebar,189.405,186,"35,229"
Petroleum Coke,600.000,20,"12,000"
Electromagnetic Control Rod,4.170,2560,"10,676"
Concrete,248.571,12,"2,983"
Time Crystal,1.737,960,"1,667"



Total: 835,077 sink points/min


### Resource utilization and shadow prices

**Shadow price** (also called *dual variable*) is the LP's answer to:
"If I had one more unit of this resource per minute, how many more sink points would I gain?"

A high shadow price = this is your bottleneck. A zero shadow price = you have spare capacity.


In [7]:
rows = []
for resource, supply in RESOURCE_SUPPLY.items():
    actual = used.get(resource, 0)
    shad   = shadow.get(resource, 0)
    rows.append({
        'Resource': resource,
        'Supply': supply if supply < 90_000 else '∞',
        'Used': round(actual, 1),
        'Slack (unused)': round(supply - actual, 1) if supply < 90_000 else 'N/A',
        'Shadow price (pts per extra unit/min)': round(shad, 2),
    })

df_res = pd.DataFrame(rows).sort_values('Shadow price (pts per extra unit/min)', ascending=False)
display(df_res.style.hide(axis='index'))


Resource,Supply,Used,Slack (unused),Shadow price (pts per extra unit/min)
coal,1440,1440.000000,-0.000000,24.000000
limestone,960,960.000000,0.000000,4.000000
iron-ore,2400,2400.000000,-0.000000,0.000000
copper-ore,960,960.000000,0.000000,0.000000
crude-oil,600,600.000000,0.000000,0.000000
water,∞,0.000000,N/A,0.000000
caterium-ore,960,960.000000,0.000000,0.000000


### Active recipes (non-zero throughput)

In [8]:
recipe_rows = []
for r, val in active:
    recipe_rows.append({
        'Recipe': r.name,
        'Building': r.building,
        'Buildings (equiv.)': round(val, 3),
    })

df_active = (
    pd.DataFrame(recipe_rows)
    .sort_values(['Building', 'Buildings (equiv.)'], ascending=[True, False])
    .reset_index(drop=True)
)
display(df_active.style.hide(axis='index'))


Recipe,Building,Buildings (equiv.)
Diamonds,accelerator,0.116000
Stun Rebar,assembler,18.940000
Magnetic Field Generator,assembler,16.098000
Modular Frame,assembler,16.013000
Reinforced Iron Plate,assembler,9.608000
Stator,assembler,8.461000
Versatile Framework,assembler,8.049000
Circuit Board,assembler,7.143000
Electromagnetic Control Rod,assembler,5.067000
Automated Wiring,assembler,4.762000


## 8. Visualization

### Points breakdown by item category

A stacked bar showing how total sink points/min break down by item tier —
this reveals which tier of the production chain is doing the most work.


In [9]:
tier_rows = []
for _, row in df_sink.iterrows():
    key = next((k for k, v in ITEMS.items() if v.name == row['Item']), None)
    tier = ITEMS[key].tier if key else -1
    tier_rows.append({'tier': tier, 'pts_per_min': row['Points/min'], 'item': row['Item']})

df_tiers = pd.DataFrame(tier_rows).sort_values('tier')

fig = go.Figure()
for _, row in df_tiers.iterrows():
    fig.add_trace(go.Bar(
        name=row['item'],
        x=[f"Tier {row['tier']}"],
        y=[row['pts_per_min']],
        hovertemplate=f"{row['item']}: {{y:,.0f}} pts/min",
    ))

fig.update_layout(
    barmode='stack',
    title='AWESOME Sink points/min by item tier',
    xaxis_title='Tier',
    yaxis_title='Points/min',
    showlegend=True,
    height=500,
)
display(HTML(fig.to_html(include_plotlyjs='cdn', full_html=False)))


### Material flow (Sankey)

In [10]:
MIN_FLOW = 1.0  # items/min threshold to keep diagram readable

node_labels: list[str] = []
node_index: dict[str, int] = {}
link_sources, link_targets, link_values, link_labels = [], [], [], []

def node_id(name: str) -> int:
    if name not in node_index:
        node_index[name] = len(node_labels)
        node_labels.append(name)
    return node_index[name]

for r, rate in active:
    if rate < 1e-4:
        continue
    for ing, qty in r.ingredients.items():
        flow = rate * qty
        if flow >= MIN_FLOW:
            label = ITEMS[ing].name if ing in ITEMS else ing
            link_sources.append(node_id(label))
            link_targets.append(node_id(r.name))
            link_values.append(flow)
            link_labels.append(f"{flow:.1f}/min")
    for prod, qty in r.products.items():
        flow = rate * qty
        if flow >= MIN_FLOW:
            label = ITEMS[prod].name if prod in ITEMS else prod
            link_sources.append(node_id(r.name))
            link_targets.append(node_id(label))
            link_values.append(flow)
            link_labels.append(f"{flow:.1f}/min")

fig2 = go.Figure(go.Sankey(
    node=dict(label=node_labels, pad=15, thickness=15),
    link=dict(source=link_sources, target=link_targets, value=link_values, label=link_labels),
))
fig2.update_layout(
    title_text="Material flow — AWESOME Sink optimization",
    height=900,
)
display(HTML(fig2.to_html(include_plotlyjs='cdn', full_html=False)))
